# KhushiDL - Heart Disease Detector

In [35]:
# Importing necessary modules
import tensorflow as tf
import pandas as pd

## Data Preprocessing

In [36]:
# Loading the data
train_data = pd.read_csv('data/train.csv', header=None)
validation_data = pd.read_csv('data/validation.csv', header=None)
test_data = pd.read_csv('data/test.csv', header=None)

In [37]:
print("Shape: ", train_data.shape)
train_data.info()

Shape:  (644, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 644 entries, 0 to 643
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       644 non-null    float64
 1   1       644 non-null    float64
 2   2       644 non-null    float64
 3   3       644 non-null    float64
 4   4       644 non-null    float64
 5   5       644 non-null    float64
 6   6       644 non-null    float64
 7   7       644 non-null    float64
 8   8       644 non-null    float64
 9   9       644 non-null    float64
 10  10      644 non-null    float64
 11  11      644 non-null    float64
 12  12      644 non-null    float64
 13  13      644 non-null    float64
dtypes: float64(14)
memory usage: 70.6 KB


In [38]:
# seperating params
train_x = train_data.iloc[:, [x for x in range(13)]]
validation_x = validation_data.iloc[:, [x for x in range(13)]]
test_x = test_data.iloc[:, [x for x in range(13)]]

In [39]:
print("Train Shape: ", train_x.shape)
print("Validation Shape: ", validation_x.shape)
print("Test Shape: ", test_x.shape)

Train Shape:  (644, 13)
Validation Shape:  (138, 13)
Test Shape:  (138, 13)


In [40]:
# sperating labels
train_y = train_data.iloc[: , [13]].copy()
validation_y = validation_data.iloc[: , [13]].copy()
test_y = test_data.iloc[: , [13]].copy()

In [41]:
print("Train Shape: ", train_y.shape)
print("Validation Shape: ", validation_y.shape)
print("Test Shape: ", test_y.shape)

Train Shape:  (644, 1)
Validation Shape:  (138, 1)
Test Shape:  (138, 1)


In [42]:
test_y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   13      138 non-null    float64
dtypes: float64(1)
memory usage: 1.2 KB


### Data Pipeline

In [43]:
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE


train_ds = (
    tf.data.Dataset.from_tensor_slices((test_x, test_y))
    .batch(BATCH)
    .shuffle(buffer_size=train_x.shape[0])
    .prefetch(AUTOTUNE)
)

validation_ds = (
    tf.data.Dataset.from_tensor_slices((validation_x, validation_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_x, test_y))
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

## Model Architecture

In [45]:
# creating the actual model
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,13)),
    
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(50, activation='relu'),
    
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1, 100)         │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1, 50)          │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1, 10)          │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,960 (27.19 KB)

 Trainable params: 6,960 (27.19 KB)

 Non-trainable params: 0 (0.00 B)

### Compiling and Training the Model

In [46]:
# Compiling the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [47]:
# EarlyStopping callback
earlyStopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)